# 02 Cohort Analysis

В этом ноутбуке рассчитываем monthly retention, D7/D30/D90 retention и размеры когорт.

## 1. Load prepared transactions

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.cohort import (
    calculate_cohort_size,
    calculate_d7_d30_d90_retention,
    calculate_monthly_retention,
)
from src.visualization import (
    plot_cohort_size,
    plot_d7_d30_d90_by_cohort,
    plot_retention_heatmap,
    plot_retention_heatmap_without_m0,
    save_dataframe,
)

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed"
IMAGES_PATH = PROJECT_ROOT / "images"
PREPARED_FILE = PROCESSED_PATH / "transactions_prepared.csv"

date_columns = ["invoice_date", "order_month", "first_purchase_date", "cohort_month"]
transactions = pd.read_csv(PREPARED_FILE, parse_dates=date_columns)

print(f"Prepared shape: {transactions.shape}")
display(transactions.head())

## 2. Build monthly cohorts

In [ ]:
retention_matrix = calculate_monthly_retention(transactions)

display(retention_matrix.round(2))

## 3. Retention heatmap

In [ ]:
plot_retention_heatmap(
    retention_matrix,
    title="Monthly Retention Heatmap",
    save_path=IMAGES_PATH / "retention_heatmap.png",
)
plt.show()

### Retention heatmap without M0

M0 всегда равен 100%, поэтому отдельная heatmap без M0 помогает лучше увидеть различия в повторном поведении после месяца первой покупки.

In [ ]:
plot_retention_heatmap_without_m0(
    retention_matrix,
    title="Monthly Retention Heatmap without M0",
    save_path=IMAGES_PATH / "retention_heatmap_without_m0.png",
)
plt.show()

## 4. D7/D30/D90 retention by cohort

In [ ]:
retention_d7_d30_d90 = calculate_d7_d30_d90_retention(transactions)

display(retention_d7_d30_d90.round(2))

plot_d7_d30_d90_by_cohort(
    retention_d7_d30_d90,
    save_path=IMAGES_PATH / "d7_d30_d90_by_cohort.png",
)
plt.show()

## 5. Cohort size analysis

In [ ]:
cohort_size = calculate_cohort_size(transactions)

display(cohort_size)

plot_cohort_size(
    cohort_size,
    save_path=IMAGES_PATH / "cohort_size_by_month.png",
)
plt.show()

## 6. Seasonal patterns and anomalies

Ниже выводятся лучшие и слабые когорты по D30 и D90 retention. Возможные причины различий требуют проверки: retention показывает динамику поведения, но не доказывает причинность.

In [ ]:
top_d30 = retention_d7_d30_d90.nlargest(5, "d30_retention")
low_d30 = retention_d7_d30_d90.nsmallest(5, "d30_retention")
top_d90 = retention_d7_d30_d90.nlargest(5, "d90_retention")
low_d90 = retention_d7_d30_d90.nsmallest(5, "d90_retention")

print("Top cohorts by D30 retention")
display(top_d30[["cohort_month", "cohort_size", "d30_retention", "d90_retention"]].round(2))

print("Lowest cohorts by D30 retention")
display(low_d30[["cohort_month", "cohort_size", "d30_retention", "d90_retention"]].round(2))

print("Top cohorts by D90 retention")
display(top_d90[["cohort_month", "cohort_size", "d30_retention", "d90_retention"]].round(2))

print("Lowest cohorts by D90 retention")
display(low_d90[["cohort_month", "cohort_size", "d30_retention", "d90_retention"]].round(2))

## 7. Save results

In [ ]:
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
IMAGES_PATH.mkdir(parents=True, exist_ok=True)

retention_matrix.to_csv(PROCESSED_PATH / "cohort_retention_matrix.csv")
save_dataframe(retention_d7_d30_d90, PROCESSED_PATH / "retention_d7_d30_d90.csv")
save_dataframe(cohort_size, PROCESSED_PATH / "cohort_size_by_month.csv")

print("Saved cohort analysis outputs.")

## 8. Key product insights

- Сравните когорты с высоким и низким D30 retention: различия могут быть связаны с сезонностью, ассортиментом, промо или составом клиентов.
- D7 retention полезен как ранний индикатор качества первого клиентского опыта.
- D90 retention показывает более устойчивое поведение, но требует учета зрелости когорт.
- Интерпретации в отчетах сформированы на основе рассчитанных метрик; при изменении данных перезапустите ноутбуки по порядку.